

```
gcloud compute ssh --zone us-west1-a ndm1-calc

chmod +x setup.sh
./setup.sh

source ~/myenv/bin/activate

sudo shutdown -h now
```



**Section 1 — Fragment Extraction**


* Load 5ZGE PDB
* Inspect residues
* Find bridging hydroxide (OH) geometrically
* Extract active site with correct residues {120, 122, 124, 189, 208, 250}
* Save raw PDB



**Section 2 — Structure Preparation**

* Add hydrogen caps (the capping code)
* Notes on Avogadro optimization
* Save final XYZ

**Section 3 — Classical QC (HF + CASSCF)**

* Build mol
* Run HF, save checkpoint
* Run CASSCF, save results
* Verify natural orbital occupancies

**Section 4 — Hamiltonian Construction**

* Extract integrals with mc.get_h1eff()
* Build Qiskit Hamiltonian
* Verify with exact sparse solver
* Save intermediates

**Section 5 — VQE**

* Load intermediates
* Run classical simulation
* Run on IBM hardware
* Compare to CASSCF reference

# Part 1 - Base energy

## Section 1 - Fragment Extraction

---



#### Load 5ZGE PDB

In [ ]:
import urllib.request
import os

url = "https://files.rcsb.org/download/5ZGE.pdb"
save_path = "/home/Daniel/gdrive/Enzyme folder/5ZGE.pdb"

if not os.path.exists(save_path):
    urllib.request.urlretrieve(url, save_path)
    print("Downloaded 5ZGE.pdb ✓")
else:
    print("5ZGE.pdb already present ✓")

5ZGE.pdb already present ✓


In [ ]:
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
structure = parser.get_structure('NDM1', '/home/Daniel/gdrive/Enzyme folder/5ZGE.pdb')

print("All non-standard residues in chain A:")
std = {'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
       'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL',
       'HOH','WAT'}
for res in structure[0]['A']:
    if res.get_resname().strip() not in std:
        atoms = list(res.get_atoms())
        print(f"  {res.get_resname():6s} {res.get_id()[1]:4d}  ({len(atoms)} atoms)")


All non-standard residues in chain A:
  ZZ7     301  (25 atoms)
  ZN      302  (1 atoms)
  ZN      303  (1 atoms)
  OH      304  (1 atoms)


In [ ]:
from Bio.PDB import PDBParser
import numpy as np

parser = PDBParser(QUIET=True)
s = parser.get_structure('NDM1', '/home/Daniel/gdrive/Enzyme folder/5ZGE.pdb')
chain = s[0]['A']
res = {r.get_id()[1]: r for r in chain}

zn1 = list(res[302].get_atoms())[0].get_vector()
zn2 = list(res[303].get_atoms())[0].get_vector()

print("ZZ7 atoms and distances to Zn1/Zn2:")
for atom in res[301].get_atoms():
    av = atom.get_vector()
    d1 = (av - zn1).norm()
    d2 = (av - zn2).norm()
    coord = ""
    if d1 < 2.8: coord += f" ← Zn1 {d1:.2f}Å"
    if d2 < 2.8: coord += f" ← Zn2 {d2:.2f}Å"
    print(f"  {atom.get_name():6s}  d(Zn1)={d1:.2f}  d(Zn2)={d2:.2f}{coord}")


ZZ7 atoms and distances to Zn1/Zn2:
  O1      d(Zn1)=5.91  d(Zn2)=4.18
  C2      d(Zn1)=5.19  d(Zn2)=2.95
  O2      d(Zn1)=5.03  d(Zn2)=2.14 ← Zn2 2.14Å
  C12     d(Zn1)=4.89  d(Zn2)=3.04
  C6      d(Zn1)=6.25  d(Zn2)=3.84
  C1      d(Zn1)=7.24  d(Zn2)=5.25
  C16     d(Zn1)=7.03  d(Zn2)=3.63
  S1      d(Zn1)=6.10  d(Zn2)=4.48
  N3      d(Zn1)=4.06  d(Zn2)=2.18 ← Zn2 2.18Å
  C13     d(Zn1)=4.58  d(Zn2)=3.27
  C14     d(Zn1)=3.89  d(Zn2)=4.31
  C15     d(Zn1)=3.24  d(Zn2)=4.81
  O4      d(Zn1)=4.10  d(Zn2)=6.04
  N1      d(Zn1)=5.13  d(Zn2)=5.59
  C3      d(Zn1)=5.84  d(Zn2)=6.25
  O3      d(Zn1)=5.73  d(Zn2)=5.98
  C4      d(Zn1)=7.15  d(Zn2)=7.68
  N2      d(Zn1)=7.86  d(Zn2)=8.59
  C5      d(Zn1)=8.17  d(Zn2)=8.01
  C7      d(Zn1)=9.28  d(Zn2)=9.30
  C8      d(Zn1)=10.36  d(Zn2)=9.94
  C9      d(Zn1)=10.46  d(Zn2)=9.42
  C10     d(Zn1)=9.50  d(Zn2)=8.15
  C11     d(Zn1)=8.30  d(Zn2)=7.35
  OXT     d(Zn1)=2.27  d(Zn2)=4.31 ← Zn1 2.27Å


#### Inspect Residues


In [ ]:
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
structure = parser.get_structure('NDM1', '/home/Daniel/gdrive/Enzyme folder/5ZGE.pdb')

print("HIS, CYS, ASP, ZN, OH in chain A:")
for residue in structure[0]['A']:
    resname = residue.get_resname().strip()
    resnum = residue.get_id()[1]
    if resname in ('HIS', 'CYS', 'ASP', 'ZN', 'OH'):
        print(f"  {resname:4s}  {resnum}")

HIS, CYS, ASP, ZN, OH in chain A:
  ASP   43
  ASP   48
  HIS   61
  ASP   66
  ASP   82
  ASP   90
  ASP   95
  ASP   96
  HIS   120
  HIS   122
  ASP   124
  ASP   130
  HIS   133
  HIS   159
  HIS   189
  ASP   192
  ASP   199
  ASP   202
  CYS   208
  ASP   212
  ASP   223
  ASP   225
  HIS   228
  HIS   250
  ASP   254
  HIS   261
  ASP   267
  ZN    302
  ZN    303
  OH    304


### Find bridging hydroxide (OH)

In [ ]:
from Bio.PDB import PDBParser
import numpy as np

parser = PDBParser(QUIET=True)
structure = parser.get_structure('NDM1', '/home/Daniel/gdrive/Enzyme folder/5ZGE.pdb')
chain_a = structure[0]['A']

zincs = [r for r in chain_a if r.get_resname().strip() == 'ZN']
zn_coords = [list(zn.get_atoms())[0].get_vector() for zn in zincs]
print(f"Zn1: residue {zincs[0].get_id()[1]}")
print(f"Zn2: residue {zincs[1].get_id()[1]}")
zn_dist = (zn_coords[0] - zn_coords[1]).norm()
print(f"Zn-Zn distance: {zn_dist:.2f} Å")

# Bridging ligand is OH (hydroxide), not HOH
bridging = [r for r in chain_a if r.get_resname().strip() == 'OH']
print(f"\nOH residues (bridging hydroxide):")
for b in bridging:
    o = list(b.get_atoms())[0]
    ov = o.get_vector()
    d1 = (ov - zn_coords[0]).norm()
    d2 = (ov - zn_coords[1]).norm()
    print(f"  OH {b.get_id()[1]}  d(Zn1)={d1:.2f} Å  d(Zn2)={d2:.2f} Å")

Zn1: residue 302
Zn2: residue 303
Zn-Zn distance: 4.60 Å

OH residues (bridging hydroxide):
  OH 304  d(Zn1)=1.97 Å  d(Zn2)=3.02 Å


##### Check

In [ ]:
import numpy as np
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
s = parser.get_structure('ndm1', '/home/Daniel/gdrive/Enzyme folder/5ZGE.pdb')
chain = s[0]['A']  # 5ZGE is a dimer — using chain A only
res = {r.get_id()[1]: r for r in chain}

# ── Step 1: find zinc ions by residue name, never by position ──────────────
zn_atoms = []
for r in chain:
    if r.get_resname().strip() == 'ZN':
        zn_atoms.append(list(r.get_atoms())[0])

assert len(zn_atoms) == 2, \
    f"Expected 2 Zn atoms in chain A, found {len(zn_atoms)}"

for i, zn in enumerate(zn_atoms):
    print(f"Zn candidate {i}: residue #{zn.get_parent().get_id()[1]}")

# ── Step 2: assign Zn1/Zn2 by chemistry, not file order ───────────────────
# Zn1 = histidine site (His120, His122, His189)
# Zn2 = cysteine site (Asp124, Cys208, His250)
assert 120 in res, "Residue 120 not found — check numbering in your extracted PDB"
assert 'NE2' in res[120], "His120 has no NE2 atom — check atom names"

his120_ne2 = res[120]['NE2'].get_vector()
d0 = (his120_ne2 - zn_atoms[0].get_vector()).norm()
d1 = (his120_ne2 - zn_atoms[1].get_vector()).norm()

if d0 > d1:
    zn_atoms = [zn_atoms[1], zn_atoms[0]]
    print("Swapped Zn order: Zn1=His-site, Zn2=Cys-site")
else:
    print("Zn order confirmed: Zn1=His-site, Zn2=Cys-site")

zn1, zn2 = zn_atoms[0], zn_atoms[1]
print(f"Zn1 residue #: {zn1.get_parent().get_id()[1]}")
print(f"Zn2 residue #: {zn2.get_parent().get_id()[1]}")

# ── Step 3: define coordination bonds ─────────────────────────────────────
LIGANDS = {
    120: ('NE2', 'Zn1', (1.9, 2.3)),
    122: ('ND1', 'Zn1', (1.9, 2.3)),
    189: ('NE2', 'Zn1', (1.9, 2.3)),
    124: ('OD2', 'Zn2', (1.9, 2.3)),
    208: ('SG',  'Zn2', (2.2, 2.5)),
    250: ('NE2', 'Zn2', (1.9, 2.3)),
}

all_ok = True
print(f'\n{"Res":<5} {"#":<6} {"Atom":<6} {"d(Zn1)":>8} {"d(Zn2)":>8} '
      f'{"Expected":>10}  Status')
print('─' * 62)

for resnum, (atname, zn_ref, (lo, hi)) in LIGANDS.items():
    if resnum not in res:
        print(f'{"???":5} {resnum:<6} {"─":6} {"MISSING RESIDUE":>30}')
        all_ok = False
        continue
    if atname not in res[resnum]:
        print(f'{res[resnum].get_resname():5} {resnum:<6} {atname:<6} '
              f'{"MISSING ATOM":>30}')
        all_ok = False
        continue

    av = res[resnum][atname].get_vector()
    d_zn1 = (av - zn1.get_vector()).norm()
    d_zn2 = (av - zn2.get_vector()).norm()
    d_ref = d_zn1 if zn_ref == 'Zn1' else d_zn2
    ok = lo <= d_ref <= hi
    if not ok:
        all_ok = False

    flag = '✓' if ok else f'✗  ({lo}–{hi} Å vs {zn_ref})'
    print(f'{res[resnum].get_resname():5} {resnum:<6} {atname:<6} '
          f'{d_zn1:8.2f} {d_zn2:8.2f} {f"vs {zn_ref}":>10}  {flag}')

# ── Step 4: bridging hydroxide (OH) ────────────────────────────────────────
# 5ZGE models the bridge as OH (hydroxide), not HOH (water)
# Geometry: Zn1-OH ~1.97 Å (tight), Zn2-OH ~3.02 Å (loose asymmetric bridge)
print('\n── Bridging hydroxide search ──────────────────────────────────')
bridge_candidates = []
for r in chain:
    if r.get_resname().strip() != 'OH':
        continue
    o_atom = next((a for a in r.get_atoms()), None)
    if o_atom is None:
        continue
    ov = o_atom.get_vector()
    d1 = (ov - zn1.get_vector()).norm()
    d2 = (ov - zn2.get_vector()).norm()
    if d1 < 3.5 and d2 < 4.0:
        bridge_candidates.append((r.get_id()[1], d1, d2))

if len(bridge_candidates) == 0:
    print('✗  No OH residue found within range of both Zn ions')
    all_ok = False
elif len(bridge_candidates) > 1:
    print(f'WARNING: {len(bridge_candidates)} OH candidates — pick closest:')
    for rnum, d1, d2 in bridge_candidates:
        print(f'  OH {rnum}: d(Zn1)={d1:.2f} Å, d(Zn2)={d2:.2f} Å')
    all_ok = False
else:
    rnum, d1, d2 = bridge_candidates[0]
    zn1_ok = 1.8 <= d1 <= 2.5
    zn2_ok = 2.5 <= d2 <= 3.5
    bw_ok = zn1_ok and zn2_ok
    if not bw_ok:
        all_ok = False
    print(f'OH {rnum}: d(Zn1)={d1:.2f} Å {"✓" if zn1_ok else "✗"}, '
          f'd(Zn2)={d2:.2f} Å {"✓" if zn2_ok else "✗"}  '
          f'{"✓ bridging hydroxide" if bw_ok else "✗ geometry unexpected"}')
    print(f'(5ZGE reference: Zn1-OH=1.97 Å, Zn2-OH=3.02 Å)')

# ── Step 5: Zn-Zn distance ─────────────────────────────────────────────────
zn_dist = (zn1.get_vector() - zn2.get_vector()).norm()
zn_ok = 4.0 <= zn_dist <= 5.0
if not zn_ok:
    all_ok = False
print(f'\nZn1–Zn2 distance: {zn_dist:.2f} Å  '
      f'(5ZGE reference: ~4.6 Å)  {"✓" if zn_ok else "✗"}')

# ── Step 6: summary ────────────────────────────────────────────────────────
print('\n' + ('═' * 62))
print('All checks passed ✓' if all_ok else
      'WARNING — review flagged items above ✗')


Zn candidate 0: residue #302
Zn candidate 1: residue #303
Zn order confirmed: Zn1=His-site, Zn2=Cys-site
Zn1 residue #: 302
Zn2 residue #: 303

Res   #      Atom     d(Zn1)   d(Zn2)   Expected  Status
──────────────────────────────────────────────────────────────
HIS   120    NE2        2.13     5.55     vs Zn1  ✓
HIS   122    ND1        2.02     6.26     vs Zn1  ✓
HIS   189    NE2        2.05     4.86     vs Zn1  ✓
ASP   124    OD2        5.06     2.11     vs Zn2  ✓
CYS   208    SG         4.34     2.37     vs Zn2  ✓
HIS   250    NE2        6.62     2.02     vs Zn2  ✓

── Bridging hydroxide search ──────────────────────────────────
OH 304: d(Zn1)=1.97 Å ✓, d(Zn2)=3.02 Å ✓  ✓ bridging hydroxide
(5ZGE reference: Zn1-OH=1.97 Å, Zn2-OH=3.02 Å)

Zn1–Zn2 distance: 4.60 Å  (5ZGE reference: ~4.6 Å)  ✓

══════════════════════════════════════════════════════════════
All checks passed ✓


### Extract Active Site & Save Raw PDB

In [ ]:
from Bio.PDB import PDBParser, PDBIO, Select

LIGAND_RESNUMS = {120, 122, 124, 189, 208, 250}

class ActiveSiteSelect(Select):
    def accept_chain(self, chain):
        return chain.id == 'A'
    def accept_residue(self, residue):
        resnum = residue.get_id()[1]
        resname = residue.get_resname().strip()
        if resnum in LIGAND_RESNUMS:
            return True
        if resname == 'ZN' and resnum in {302, 303}:
            return True
        if resname == 'OH' and resnum == 304:
            return True
        return False

parser = PDBParser(QUIET=True)
structure = parser.get_structure('NDM1', '/home/Daniel/gdrive/Enzyme folder/5ZGE.pdb')

io = PDBIO()
io.set_structure(structure)
raw_path = '/home/Daniel/gdrive/Enzyme folder/ndm1_active_site_raw.pdb'
io.save(raw_path, ActiveSiteSelect())
print(f"Saved {raw_path} ✓")

# Verify
s2 = parser.get_structure('frag', raw_path)
for res in s2.get_residues():
    atoms = list(res.get_atoms())
    print(f"  {res.get_resname():4s} {res.get_id()[1]:4d}  ({len(atoms)} atoms)")

Saved /home/Daniel/gdrive/Enzyme folder/ndm1_active_site_raw.pdb ✓
  HIS   120  (10 atoms)
  HIS   122  (10 atoms)
  ASP   124  (8 atoms)
  HIS   189  (10 atoms)
  CYS   208  (6 atoms)
  HIS   250  (10 atoms)
  ZN    302  (1 atoms)
  ZN    303  (1 atoms)
  OH    304  (1 atoms)


## Section 2 - Structure Preparation

---

#### Add Hydrogen Caps

Each residue keeps its full backbone (N, CA, C, O) + sidechain. One H cap is placed on N along the N→(away from CA) direction at 1.01 Å, replacing the bond to the preceding residue's carbonyl. The C-terminus C=O is left as a natural formyl-like cap. CYS208 SG and ASP124 OD1/OD2 are deprotonated. Bridging OH gets one H placed away from the Zn–Zn midpoint.

In [ ]:
import numpy as np
from Bio.PDB import PDBParser

def normalize(v):
    return v / np.linalg.norm(v)

parser = PDBParser(QUIET=True)
structure = parser.get_structure('frag', '/home/Daniel/gdrive/Enzyme folder/ndm1_active_site_raw.pdb')

atoms_out = []  # list of (element, x, y, z)
zn_positions = []
oh_position = None

AA_ELEM = {
    'N': 'N', 'CA': 'C', 'C': 'C', 'O': 'O', 'CB': 'C',
    'CG': 'C', 'ND1': 'N', 'CD2': 'C', 'CE1': 'C', 'NE2': 'N',  # HIS
    'OD1': 'O', 'OD2': 'O',                                        # ASP
    'SG': 'S',                                                      # CYS
}

for residue in structure.get_residues():
    resname = residue.get_resname().strip()
    atom_dict = {a.get_name().strip(): a.get_vector().get_array() for a in residue.get_atoms()}

    if resname == 'ZN':
        pos = list(atom_dict.values())[0]
        atoms_out.append(('Zn', *pos))
        zn_positions.append(pos)

    elif resname == 'OH':
        pos = list(atom_dict.values())[0]
        oh_position = pos
        atoms_out.append(('O', *pos))

    else:  # HIS, ASP, CYS
        for name, pos in atom_dict.items():
            elem = AA_ELEM.get(name, name[0])
            atoms_out.append((elem, *pos))
        # H cap at N: place H on the far side of N from CA
        n_pos  = atom_dict['N']
        ca_pos = atom_dict['CA']
        h_pos  = n_pos + 1.01 * normalize(n_pos - ca_pos)
        atoms_out.append(('H', *h_pos))

# H on bridging OH: place along O away from Zn midpoint
zn_mid = (zn_positions[0] + zn_positions[1]) / 2
h_oh   = oh_position + 0.97 * normalize(oh_position - zn_mid)
atoms_out.append(('H', *h_oh))

out_path = '/home/Daniel/gdrive/Enzyme folder/ndm1_active_site_capped.xyz'
with open(out_path, 'w') as f:
    f.write(f"{len(atoms_out)}\nNDM-1 active site capped (5ZGE chain A, charge=+1)\n")
    for elem, x, y, z in atoms_out:
        f.write(f"{elem:<4s} {x:12.6f} {y:12.6f} {z:12.6f}\n")

print(f"Saved {out_path} ({len(atoms_out)} atoms) ✓")

Saved /home/Daniel/gdrive/Enzyme folder/ndm1_active_site_capped.xyz (64 atoms) ✓


#### Avogadro Optimization

Load `ndm1_active_site_capped.xyz` in Avogadro. Run **Extensions → Molecular Mechanics → Optimize Geometry** (UFF force field) to relax the H cap positions and any strained bonds introduced by the cut. Save the result as `ndm1_active_site_optimized.xyz`. Then save a copy as `ndm1_active_site_hydroxide.xyz` — this is the file that goes into PySCF.

## Section 3 - Classical QC

---

#### Build mol

In [ ]:
from pyscf import gto, scf
import numpy as np

xyz_path = '/home/Daniel/gdrive/Enzyme folder/ndm1_active_site_hydroxide.xyz'

with open(xyz_path) as f:
    lines = f.readlines()
n_atoms = int(lines[0])
atom_lines = lines[2:2+n_atoms]

atom_str = ''
for line in atom_lines:
    parts = line.split()
    elem = parts[0]
    x, y, z = parts[1], parts[2], parts[3]
    atom_str += f'{elem} {x} {y} {z}; '

mol = gto.Mole()
mol.atom  = atom_str
mol.basis = 'def2-svp'
mol.charge = 1
mol.spin   = 0
mol.verbose = 4
mol.build()

print(f"Electrons: {mol.nelectron}")
print(f"Basis functions: {mol.nao}")

System: uname_result(system='Linux', node='ndm1-calc', release='6.1.0-49-cloud-amd64', version='#1 SMP PREEMPT_DYNAMIC Debian 6.1.174-1 (2026-05-26)', machine='x86_64')  Threads 8
Python 3.11.2 (main, Apr  8 2026, 01:58:00) [GCC 12.2.0]
numpy 2.4.6  scipy 1.17.1  h5py 3.16.0
Date: Fri Jun 19 21:48:52 2026
PySCF version 2.13.1
PySCF path  /home/Daniel/myenv/lib/python3.11/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 64
[INPUT] num. electrons = 438
[INPUT] charge = 1
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 N      3.639946509700  44.709269693200 117.920525520300 AA    6.878502011400  84.488274949465 222.837497698152 Bohr   0.0
[INPUT]  2 C      4.484745192400  43.895133803200 117.058786440700 AA    8.474940152096  82.949781089186 221.209

#### Verify electron count

In [ ]:
mol.build()
print('Electrons:', mol.nelectron)
print('Alpha:', mol.nelec[0], 'Beta:', mol.nelec[1])
assert mol.nelectron % 2 == 0, "ODD ELECTRON COUNT — wrong charge or spin"
assert mol.nelec[0] == mol.nelec[1], "SPIN CONTAMINATION — not a singlet"

Electrons: 438
Alpha: 219 Beta: 219


#### DF-B3LYP (next 2 cells)


In [ ]:
import psutil
from pyscf import dft, scf

avail_mb = psutil.virtual_memory().available / 1e6

dm0 = scf.RHF(mol).from_chk('/home/Daniel/ndm1_b3lyp_def2svp.chk')

mf = dft.RKS(mol).density_fit()
mf.xc                 = 'b3lyp'
mf.with_df.auxbasis   = 'def2-universal-jkfit'
mf.with_df._cderi     = '/home/Daniel/ndm1_df_eri_svp.h5'

mf.chkfile     = '/home/Daniel/ndm1_b3lyp_def2svp.chk'
mf.max_memory  = int(min(24000, avail_mb * 0.8))
mf.max_cycle   = 500
mf.diis_space  = 8
mf.level_shift = 0.5
mf.damp        = 0.5
mf.verbose     = 4

mf.kernel(dm0=dm0)




******** <class 'pyscf.df.df_jk.DFRKS'> ********
method = DFRKS
initial guess = minao
damping factor = 0.5
level_shift factor = 0.5
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-09
SCF conv_tol_grad = None
SCF max_cycles = 500
direct_scf = False
chkfile to save SCF result = /home/Daniel/ndm1_b3lyp_def2svp.chk
max_memory 23861 MB (current use 174 MB)
XC library pyscf.dft.libxc version 7.0.0
    S. Lehtola, C. Steigemann, M. J.T. Oliveira, and M. A.L. Marques.,  SoftwareX 7, 1–5 (2018)
XC functionals = b3lyp
    P. J. Stephens, F. J. Devlin, C. F. Chabalowski, and M. J. Frisch.,  J. Phys. Chem. 98, 11623 (1994)
radial grids: 
    Treutler-Ahlrichs [JCP 102, 346 (1995); DOI:10.1063/1.469408] (M4) radial grids
    
becke partition: Becke, JCP 88, 2547 (1988); DOI:10.1063/1.454033
pruning grids: <function nwchem_prune at 0x7f61786d3240>
grids dens level: 3
symmetrized grids: False
atomic radii adjust function: <function treutler_

KeyboardInterrupt: 

In [ ]:
import psutil
from pyscf import dft, scf

avail_mb = psutil.virtual_memory().available / 1e6

dm0 = scf.RHF(mol).from_chk('/home/Daniel/ndm1_b3lyp_def2svp.chk')

mf = dft.RKS(mol).density_fit()
mf.xc                 = 'b3lyp'
mf.with_df.auxbasis   = 'def2-universal-jkfit'
mf.with_df._cderi     = '/home/Daniel/ndm1_df_eri_svp.h5'
mf.chkfile            = '/home/Daniel/ndm1_b3lyp_def2svp.chk'
mf.max_memory         = int(min(24000, avail_mb * 0.8))
mf.verbose            = 4

mf = mf.newton()
mf.max_cycle = 50

mf.kernel(dm0=dm0)






******** <class 'pyscf.soscf.newton_ah.SecondOrderDFRKS'> ********
method = SecondOrderDFRKS
initial guess = minao
damping factor = 0
level_shift factor = 0
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-09
SCF conv_tol_grad = None
SCF max_cycles = 50
direct_scf = False
chkfile to save SCF result = /home/Daniel/ndm1_b3lyp_def2svp.chk
max_memory 22826 MB (current use 1335 MB)
XC library pyscf.dft.libxc version 7.0.0
    S. Lehtola, C. Steigemann, M. J.T. Oliveira, and M. A.L. Marques.,  SoftwareX 7, 1–5 (2018)
XC functionals = b3lyp
    P. J. Stephens, F. J. Devlin, C. F. Chabalowski, and M. J. Frisch.,  J. Phys. Chem. 98, 11623 (1994)
radial grids: 
    Treutler-Ahlrichs [JCP 102, 346 (1995); DOI:10.1063/1.469408] (M4) radial grids
    
becke partition: Becke, JCP 88, 2547 (1988); DOI:10.1063/1.454033
pruning grids: <function nwchem_prune at 0x7f61786d3240>
grids dens level: 3
symmetrized grids: False
atomic radii adjust fu

np.float64(-6579.779498560714)

In [ ]:
print(f"HF converged: {mf.converged}")
print(f"HF energy:    {mf.e_tot:.8f} Eh")

mo_e = mf.mo_energy
nocc = mol.nelectron // 2
homo = mo_e[nocc - 1]
lumo = mo_e[nocc]
print(f"HOMO: {homo:.6f} Eh  ({homo * 27.2114:.3f} eV)")
print(f"LUMO: {lumo:.6f} Eh  ({lumo * 27.2114:.3f} eV)")
print(f"Gap:  {(lumo - homo) * 27.2114:.3f} eV")

HF converged: True
HF energy:    -6579.77949856 Eh
HOMO: -0.319434 Eh  (-8.692 eV)
LUMO: -0.309277 Eh  (-8.416 eV)
Gap:  0.276 eV


#### Verify HOMO-LUMO gap is physically reasonable after convergence.
For the dizinc fragment:
* Gap below 0.1 eV: you're in a near-degenerate situation, active space selection will be unreliable
* Gap 0.5–3.0 eV: normal for a transition metal complex, proceed
* Gap above 5 eV: suspiciously large, may indicate wrong charge or spin state

In [ ]:
mo_energies = mf.mo_energy
mo_occ = mf.mo_occ
homo = max(mo_energies[mo_occ > 0])
lumo = min(mo_energies[mo_occ == 0])
gap_ev = (lumo - homo) * 27.2114  # Hartree to eV
print(f'HOMO: {homo*27.2114:.3f} eV')
print(f'LUMO: {lumo*27.2114:.3f} eV')
print(f'Gap:  {gap_ev:.3f} eV')
assert gap_ev > 0.1, f"Gap too small ({gap_ev:.3f} eV) — near-degenerate, check active space"

HOMO: -8.692 eV
LUMO: -8.416 eV
Gap:  0.276 eV


#### Verification

In [ ]:
from pyscf import scf as pyscf_scf
import numpy as np

# ── Check 1: convergence flag ─────────────────────────────────────────────
assert mf.converged, "Not converged"
print("Converged: ✓")

# ── Check 2: electron count ───────────────────────────────────────────────
mo_coeff = pyscf_scf.chkfile.load('/home/Daniel/ndm1_b3lyp_def2svp.chk', 'scf/mo_coeff')
mo_occ   = pyscf_scf.chkfile.load('/home/Daniel/ndm1_b3lyp_def2svp.chk', 'scf/mo_occ')
nelec    = mo_occ.sum()
print(f"Electrons (sum of mo_occ): {nelec:.1f}  (expected {mol.nelectron})")
assert abs(nelec - mol.nelectron) < 0.01, f"Electron count wrong: {nelec}"

# ── Check 3: HOMO-LUMO gap ────────────────────────────────────────────────
mo_e       = mf.mo_energy
mo_occ_arr = mf.mo_occ
homo   = max(mo_e[mo_occ_arr > 0])
lumo   = min(mo_e[mo_occ_arr == 0])
gap_ev = (lumo - homo) * 27.2114
print(f"HOMO: {homo * 27.2114:.3f} eV")
print(f"LUMO: {lumo * 27.2114:.3f} eV")
print(f"Gap:  {gap_ev:.3f} eV")
assert gap_ev > 0, "Negative gap — orbital crossing, something is wrong"

# ── Check 4: energy is reasonable ────────────────────────────────────────
assert -7000 < mf.e_tot < -6000, f"Energy out of range: {mf.e_tot:.2f}"
print(f"Energy: {mf.e_tot:.8f} Hartree  ✓")

# ── Check 5: orbital occupations are integer (closed-shell) ──────────────
unique_occ = np.unique(np.round(mo_occ_arr, 6))
print(f"Unique occupations: {unique_occ}")
assert set(np.round(unique_occ, 1)).issubset({0.0, 2.0}), \
    "Non-integer occupations — unexpected open-shell or fractional occupation"
print("Occupations: ✓")

print("\nAll checks passed ✓")


Converged: ✓
Electrons (sum of mo_occ): 438.0  (expected 438)
HOMO: -8.692 eV
LUMO: -8.416 eV
Gap:  0.276 eV
Energy: -6579.77949856 Hartree  ✓
Unique occupations: [0. 2.]
Occupations: ✓

All checks passed ✓


#### Rebuilding mol and reload mf from the checkpoint after kernal shutdown

In [ ]:
from pyscf import gto, dft, scf as pyscf_scf, mcscf
import psutil

avail_mb = psutil.virtual_memory().available / 1e6

# Rebuild mol
xyz_path = '/home/Daniel/gdrive/Enzyme folder/ndm1_active_site_hydroxide.xyz'
with open(xyz_path) as f:
    lines = f.readlines()
n_atoms = int(lines[0])
atom_str = ''
for line in lines[2:2+n_atoms]:
    parts = line.split()
    atom_str += f'{parts[0]} {parts[1]} {parts[2]} {parts[3]}; '

mol = gto.Mole()
mol.atom    = atom_str
mol.basis   = 'def2-svp'
mol.charge  = 1
mol.spin    = 0
mol.verbose = 4
mol.build()

# Rebuild mf and load MOs from checkpoint
mf = dft.RKS(mol).density_fit()
mf.xc               = 'b3lyp'
mf.with_df.auxbasis = 'def2-universal-jkfit'
mf.with_df._cderi   = '/home/Daniel/ndm1_df_eri_svp.h5'
mf.chkfile          = '/home/Daniel/ndm1_b3lyp_def2svp.chk'
mf.max_memory       = int(min(24000, avail_mb * 0.8))

mf.mo_coeff  = pyscf_scf.chkfile.load('/home/Daniel/ndm1_b3lyp_def2svp.chk', 'scf/mo_coeff')
mf.mo_occ    = pyscf_scf.chkfile.load('/home/Daniel/ndm1_b3lyp_def2svp.chk', 'scf/mo_occ')
mf.mo_energy = pyscf_scf.chkfile.load('/home/Daniel/ndm1_b3lyp_def2svp.chk', 'scf/mo_energy')
mf.e_tot     = float(pyscf_scf.chkfile.load('/home/Daniel/ndm1_b3lyp_def2svp.chk', 'scf/e_tot'))

print(f"Loaded B3LYP energy: {mf.e_tot:.8f} Hartree")
print(f"MO shape: {mf.mo_coeff.shape}")
print(f"Electrons: {mf.mo_occ.sum():.0f}")


System: uname_result(system='Linux', node='ndm1-calc', release='6.1.0-49-cloud-amd64', version='#1 SMP PREEMPT_DYNAMIC Debian 6.1.174-1 (2026-05-26)', machine='x86_64')  Threads 8
Python 3.11.2 (main, Apr  8 2026, 01:58:00) [GCC 12.2.0]
numpy 2.4.6  scipy 1.17.1  h5py 3.16.0
Date: Sun Jun 21 23:35:47 2026
PySCF version 2.13.1
PySCF path  /home/Daniel/myenv/lib/python3.11/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 64
[INPUT] num. electrons = 438
[INPUT] charge = 1
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 N      3.639946509700  44.709269693200 117.920525520300 AA    6.878502011400  84.488274949465 222.837497698152 Bohr   0.0
[INPUT]  2 C      4.484745192400  43.895133803200 117.058786440700 AA    8.474940152096  82.949781089186 221.209

### Inspecting the B3LYP frontier orbitals

In [ ]:
import numpy as np

mo_e   = mf.mo_energy
mo_occ = mf.mo_occ
nocc   = np.sum(mo_occ > 0)

print("Frontier MO energies (eV):")
print("Occupied (HOMO-5 to HOMO):")
for i in range(nocc-6, nocc):
    print(f"  MO {i:3d} (occ={mo_occ[i]:.0f}): {mo_e[i]*27.2114:.3f} eV")
print("Virtual (LUMO to LUMO+5):")
for i in range(nocc, nocc+6):
    print(f"  MO {i:3d} (occ={mo_occ[i]:.0f}): {mo_e[i]*27.2114:.3f} eV")


Frontier MO energies (eV):
Occupied (HOMO-5 to HOMO):
  MO 213 (occ=2): -9.716 eV
  MO 214 (occ=2): -9.593 eV
  MO 215 (occ=2): -9.428 eV
  MO 216 (occ=2): -9.268 eV
  MO 217 (occ=2): -9.193 eV
  MO 218 (occ=2): -8.692 eV
Virtual (LUMO to LUMO+5):
  MO 219 (occ=0): -8.416 eV
  MO 220 (occ=0): -8.346 eV
  MO 221 (occ=0): -8.286 eV
  MO 222 (occ=0): -8.243 eV
  MO 223 (occ=0): -8.188 eV
  MO 224 (occ=0): -7.989 eV


In [ ]:
print("MO 208-224 energies:")
for i in range(208, 225):
    tag = " ← HOMO" if i == 218 else (" ← LUMO" if i == 219 else "")
    print(f"  MO {i} (occ={int(mf.mo_occ[i])}): {mf.mo_energy[i]*27.2114:.3f} eV{tag}")


MO 208-224 energies:
  MO 208 (occ=2): -9.981 eV
  MO 209 (occ=2): -9.942 eV
  MO 210 (occ=2): -9.831 eV
  MO 211 (occ=2): -9.760 eV
  MO 212 (occ=2): -9.744 eV
  MO 213 (occ=2): -9.716 eV
  MO 214 (occ=2): -9.593 eV
  MO 215 (occ=2): -9.428 eV
  MO 216 (occ=2): -9.268 eV
  MO 217 (occ=2): -9.193 eV
  MO 218 (occ=2): -8.692 eV ← HOMO
  MO 219 (occ=0): -8.416 eV ← LUMO
  MO 220 (occ=0): -8.346 eV
  MO 221 (occ=0): -8.286 eV
  MO 222 (occ=0): -8.243 eV
  MO 223 (occ=0): -8.188 eV
  MO 224 (occ=0): -7.989 eV


#### CASCI(7,2)
A complete active space with 7 orbitals and 2 electrons using MOs 218-224 (the frotier orbitals nearest HOMO). This gives a more accurate, correlated wavefuntion for the active space, and the 1-electron (h1e), 2-electron (h2e), and core energy integrals are extracted and saved as .npy files.

In [ ]:
from pyscf import mcscf, ao2mo
import numpy as np

ncas    = 7
nelecas = 2

mc = mcscf.CASCI(mf, ncas, nelecas)
mo = mc.sort_mo(list(range(218, 225)))
mc.verbose = 4
mc.kernel(mo)

h1e, ecore = mc.get_h1eff()
h2e = ao2mo.restore(1, mc.get_h2eff(), ncas)

np.save('/home/Daniel/ndm1_h1e.npy', h1e)
np.save('/home/Daniel/ndm1_h2e.npy', h2e)
np.save('/home/Daniel/ndm1_ecore.npy', np.array([ecore]))
print(f"CASCI energy: {mc.e_tot:.8f} Eh")
print("Saved h1e, h2e, ecore ✓")



******** <class 'pyscf.mcscf.df.DFCASCI'> ********
CAS (1e+1e, 7o), ncore = 218, nvir = 646
natorb = False
canonicalization = True
sorting_mo_energy = False
max_memory 24000 MB (current use 161 MB)
******** <class 'pyscf.fci.direct_spin1.FCISolver'> ********
max. cycles = 200
conv_tol = 1e-08
davidson only = False
linear dependence = 1e-12
level shift = 0.001
max iter space = 12
max_memory 4000 MB
nroots = 1
pspace_size = 400
spin = None


DFCASCI/DFCASSCF: density fitting for JK matrix and 2e integral transformation
Density matrix diagonal elements [1.87657869e+00 6.35827461e-03 1.41444507e-02 1.41525451e-03
 4.42023625e-02 3.37211361e-03 5.39288573e-02]
CASCI converged
CASCI E = -6558.83741602724  E(CI) = -1.25374911372728  S^2 = 0.0000000
CASCI energy: -6558.83741603 Eh
Saved h1e, h2e, ecore ✓


#### Verify natural orbital occupancies

In [ ]:
import numpy as np

nocc = mol.nelectron // 2  # = 219

noons, nos = mcscf.addons.make_natural_orbitals(mc)

print(f"CASCI energy: {mc.e_tot:.8f} Eh")
print(f"\nNatural orbital occupancies (active space, {ncas} orbitals):")
for i, occ in enumerate(noons[nocc - ncas//2 : nocc + ncas//2 + 2]):
    flag = " <-- strongly correlated" if 0.1 < occ < 1.9 else ""
    print(f"  NO {i:2d}: {occ:.4f}{flag}")


CASCI energy: -6558.83741603 Eh

Natural orbital occupancies (active space, 7 orbitals):
  NO  0: 2.0000
  NO  1: 2.0000
  NO  2: 1.9983
  NO  3: 0.0015
  NO  4: 0.0002
  NO  5: 0.0000
  NO  6: 0.0000
  NO  7: 0.0000


## Section 4 - Hamiltonian Construction

---

#### Extract integrals & build Qiskit Hamiltonian
The saved integrals (h1e, h2e, ecore) are loaded and repackaged into a Qiskit ElectronicEnergy Hamiltonian object. The JordanWignerMapper then converts the fermionic second-quantized Hamiltonian to a qubit operator (a sum of Pauli string operators) — a 14-qubit operator (one qubit per spin-orbital for 7 orbitals × 2 spins). The result is verified by diagonalizing the Hamiltonian matrix exactly in the 2-electron subspace (selecting the 91 of 16,384 basis states with exactly 2 electrons set in binary representation), and confirming the ground state energy matches the CASCI reference to sub-milliHartree precision.

In [ ]:
import numpy as np
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy
from qiskit_nature.second_q.mappers import JordanWignerMapper

# Load saved integrals
h1e   = np.load('/home/Daniel/ndm1_h1e.npy')
h2e   = np.load('/home/Daniel/ndm1_h2e.npy')   # PySCF chemist's (ij|kl)
ecore = float(np.load('/home/Daniel/ndm1_ecore.npy')[0])


print(f"h1e: {h1e.shape},  h2e: {h2e.shape},  ecore: {ecore:.6f} Eh")

# Qiskit Nature expects physicist's notation <ij|kl> = chemist's (ik|jl)
h2e_phys = h2e.transpose(0, 2, 1, 3)

hamiltonian = ElectronicEnergy.from_raw_integrals(h1e, h2e_phys)
hamiltonian.nuclear_repulsion_energy = ecore

qubit_op = JordanWignerMapper().map(hamiltonian.second_q_op())

print(f"Qubits:      {qubit_op.num_qubits}")
print(f"Pauli terms: {len(qubit_op)}")


h1e: (7, 7),  h2e: (7, 7, 7, 7),  ecore: -6557.583667 Eh
Qubits:      14
Pauli terms: 3382


#### Verify with exact sparse solver

In [ ]:
from pyscf import fci
import numpy as np

h1e   = np.load('/home/Daniel/ndm1_h1e.npy')
h2e   = np.load('/home/Daniel/ndm1_h2e.npy')
ecore = np.load('/home/Daniel/ndm1_ecore.npy')[0]

# Exact FCI in active space (sparse solver — reference for VQE)
e_fci, ci = fci.direct_spin0.kernel(h1e, h2e, ncas, nelecas)
e_fci_total = e_fci + ecore

print(f"CASSCF energy:      {mc.e_tot:.8f} Eh")
print(f"FCI (active space): {e_fci_total:.8f} Eh")
print(f"Difference:         {(e_fci_total - mc.e_tot)*1000:.4f} mEh")

CASSCF energy:      -6558.83741603 Eh
FCI (active space): -6558.83741603 Eh
Difference:         0.0000 mEh


## Section 5 - VQE

---


#### Reload integrals and rebuild Hamiltonian

In [ ]:
import numpy as np
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy
from qiskit_nature.second_q.mappers import JordanWignerMapper

h1e   = np.load('/home/Daniel/ndm1_h1e.npy')
h2e   = np.load('/home/Daniel/ndm1_h2e.npy')
ecore = float(np.load('/home/Daniel/ndm1_ecore.npy')[0])

h2e_phys = h2e.transpose(0, 2, 1, 3)
hamiltonian = ElectronicEnergy.from_raw_integrals(h1e, h2e_phys)
hamiltonian.nuclear_repulsion_energy = ecore

qubit_op = JordanWignerMapper().map(hamiltonian.second_q_op())
print(f"Qubits: {qubit_op.num_qubits}, Terms: {len(qubit_op)}")


Qubits: 14, Terms: 3382


#### Simulation

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy
from qiskit_nature.second_q.mappers import JordanWignerMapper

h1e   = np.load('/home/Daniel/ndm1_h1e.npy')
h2e   = np.load('/home/Daniel/ndm1_h2e.npy')
ecore = float(np.load('/home/Daniel/ndm1_ecore.npy')[0])

hamiltonian = ElectronicEnergy.from_raw_integrals(h1e, h2e)
hamiltonian.nuclear_repulsion_energy = ecore
qubit_op = JordanWignerMapper().map(hamiltonian.second_q_op())

# Restrict to 2-electron subspace (91 of 16384 basis states)
n_qubits = qubit_op.num_qubits
basis_2e = [i for i in range(2**n_qubits) if bin(i).count('1') == 2]
H_full = qubit_op.to_matrix(sparse=True)
H_sub  = H_full[np.ix_(basis_2e, basis_2e)]
evals  = sp.linalg.eigsh(H_sub, k=1, which='SA', return_eigenvectors=False)

e_gs = float(evals[0]) + ecore
print(f"Exact ground state:  {e_gs:.8f} Eh")
print(f"CASCI reference:     {-6558.83741603:.8f} Eh")
print(f"Difference:          {(e_gs - (-6558.83741603))*1000:.4f} mEh")
print(f"\nHamiltonian verified ✓")
print(f"Qubits: {n_qubits}, 2e subspace: {len(basis_2e)} states")


Exact ground state:  -6558.83741603 Eh
CASCI reference:     -6558.83741603 Eh
Difference:          0.0000 mEh

Hamiltonian verified ✓
Qubits: 14, 2e subspace: 91 states


#### Finding optimal parameters

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import COBYLA

# Rebuild penalized Hamiltonian
n_qubits = 14
n_target = 2

N_terms = [('I'*n_qubits, n_qubits / 2.0)]
for i in range(n_qubits):
    N_terms.append(('I'*i + 'Z' + 'I'*(n_qubits-1-i), -0.5))
N_op = SparsePauliOp.from_list(N_terms)
N2_op = (N_op @ N_op).simplify()
penalty = (N2_op - 2*n_target*N_op
           + SparsePauliOp.from_list([('I'*n_qubits, float(n_target**2))])).simplify()
penalized_op = (qubit_op + 5.0 * penalty).simplify()

# Ansatz with HF initial state
hf_prep = QuantumCircuit(n_qubits)
hf_prep.x(0)
hf_prep.x(7)
ansatz = EfficientSU2(n_qubits, reps=2, entanglement='linear', initial_state=hf_prep)

# Classical VQE to find optimal parameters
energy_log = []
def callback(nfev, x, fx, metadata):
    energy_log.append(fx)
    if nfev % 20 == 0:
        print(f"  iter {nfev:3d}  E = {fx + ecore:.6f} Eh")

estimator = StatevectorEstimator()
vqe = VQE(estimator, ansatz, COBYLA(maxiter=300),
          initial_point=np.zeros(ansatz.num_parameters),
          callback=callback)
result_sim = vqe.compute_minimum_eigenvalue(penalized_op)

optimal_params = result_sim.optimal_point
e_classical = result_sim.eigenvalue.real + ecore
print(f"\nClassical VQE:  {e_classical:.8f} Eh")
print(f"CASCI ref:      {-6558.83741603:.8f} Eh")
print(f"Difference:     {(e_classical - (-6558.83741603))*1000:.4f} mEh")
np.save('/home/Daniel/ndm1_vqe_params.npy', optimal_params)
print(f"Optimal params saved ✓  ({len(optimal_params)} parameters)")


/tmp/ipykernel_3299/744470431.py:26: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(n_qubits, reps=2, entanglement='linear', initial_state=hf_prep)


  iter  20  E = -6391.520324 Eh
  iter  40  E = -6421.313769 Eh
  iter  60  E = -6421.939218 Eh
  iter  80  E = -6437.959200 Eh
  iter 100  E = -6482.407250 Eh
  iter 120  E = -6501.185768 Eh
  iter 140  E = -6503.999609 Eh
  iter 160  E = -6505.532937 Eh
  iter 180  E = -6513.923334 Eh
  iter 200  E = -6514.179178 Eh
  iter 220  E = -6506.974199 Eh
  iter 240  E = -6517.267084 Eh
  iter 260  E = -6524.315872 Eh
  iter 280  E = -6532.945753 Eh
  iter 300  E = -6534.874192 Eh

Classical VQE:  -6534.87419244 Eh
CASCI ref:      -6558.83741603 Eh
Difference:     23963.2236 mEh
Optimal params saved ✓  (84 parameters)


##### Timing check

In [ ]:
from qiskit import transpile
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import COBYLA
import numpy as np

# Pre-decompose UCCSD to basic gates ONCE — eliminates per-iteration overhead
ansatz_decomposed = transpile(ansatz, basis_gates=['cx', 'u'], optimization_level=1)
print(f"Decomposed depth: {ansatz_decomposed.depth()}, 2Q gates: {ansatz_decomposed.num_nonlocal_gates()}")

# Time a single evaluation on the decomposed circuit
import time
bound = ansatz_decomposed.assign_parameters(np.zeros(ansatz_decomposed.num_parameters))
t0 = time.time()
StatevectorEstimator().run([(bound, qubit_op)]).result()
print(f"Single eval time: {time.time()-t0:.2f}s")


Decomposed depth: 5089, 2Q gates: 4596
Single eval time: 3.05s


In [ ]:
import time
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp

bound = ansatz.assign_parameters(np.zeros(ansatz.num_parameters))
t0 = time.time()
job = StatevectorEstimator().run([(bound, qubit_op)])
job.result()
print(f"Single evaluation: {time.time()-t0:.2f}s")


Single evaluation: 0.49s


#### UCCSD-VQE simulation: uses a physics motivated UCCSD ansatz (Unitary Coupled Cluster Singles and Doubles)
* Preserves particle number naturally (no penalty term needed)
* Has a physically motivated starting point (HF = all params zero)
* For a near-HF system, should converge in ~20-30 iterations

In [ ]:
energy_log = []
def callback(nfev, x, fx, metadata):
    energy_log.append(fx + ecore)
    if nfev % 10 == 0:
        print(f"  iter {nfev:3d}  E = {fx + ecore:.8f} Eh")

vqe = VQE(StatevectorEstimator(), ansatz_decomposed,
          COBYLA(maxiter=300, rhobeg=0.01),
          initial_point=np.zeros(ansatz_decomposed.num_parameters),
          callback=callback)

result_sim = vqe.compute_minimum_eigenvalue(qubit_op)
optimal_params = result_sim.optimal_point
e_vqe_sim = result_sim.eigenvalue.real + ecore

print(f"\nUCCSD-VQE (classical): {e_vqe_sim:.8f} Eh")
print(f"CASCI ref:             {-6558.83741603:.8f} Eh")
print(f"Difference:            {(e_vqe_sim - (-6558.83741603))*1000:.4f} mEh")
np.save('/home/Daniel/ndm1_uccsd_params.npy', optimal_params)


  iter  10  E = -6558.81931242 Eh
  iter  20  E = -6558.81996298 Eh
  iter  30  E = -6558.81994729 Eh
  iter  40  E = -6558.82007649 Eh
  iter  50  E = -6558.82147568 Eh
  iter  60  E = -6558.82904730 Eh
  iter  70  E = -6558.83056240 Eh
  iter  80  E = -6558.83137552 Eh
  iter  90  E = -6558.83162210 Eh
  iter 100  E = -6558.83208057 Eh
  iter 110  E = -6558.83214254 Eh
  iter 120  E = -6558.83399443 Eh
  iter 130  E = -6558.83461104 Eh
  iter 140  E = -6558.83498692 Eh
  iter 150  E = -6558.83518569 Eh
  iter 160  E = -6558.83571681 Eh
  iter 170  E = -6558.83573207 Eh
  iter 180  E = -6558.83619012 Eh
  iter 190  E = -6558.83636409 Eh
  iter 200  E = -6558.83637367 Eh
  iter 210  E = -6558.83632809 Eh
  iter 220  E = -6558.83664249 Eh
  iter 230  E = -6558.83669157 Eh
  iter 240  E = -6558.83680526 Eh
  iter 250  E = -6558.83688429 Eh
  iter 260  E = -6558.83688529 Eh
  iter 270  E = -6558.83689283 Eh
  iter 280  E = -6558.83690923 Eh
  iter 290  E = -6558.83701631 Eh
  iter 300  E 

#### Finding backend

In [ ]:
service = QiskitRuntimeService(channel="ibm_quantum_platform")

# Pick least-busy backend with ≥14 qubits and real hardware
backends = service.backends(
    filters=lambda b: b.configuration().n_qubits >= 14
                   and not b.configuration().simulator
                   and b.status().operational
)
# Sort by queue length
backend = sorted(backends, key=lambda b: b.status().pending_jobs)[0]
print(f"Selected backend: {backend.name}")
print(f"  Qubits:       {backend.configuration().n_qubits}")
print(f"  Pending jobs: {backend.status().pending_jobs}")


qiskit_runtime_service.__init__:WARNING:2026-06-23 00:23:23,683: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-06-23 00:23:23,685: Loading instance: open-instance, plan: open


Selected backend: ibm_kingston
  Qubits:       156
  Pending jobs: 0


#### Running on quantum computer

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import efficient_su2
from qiskit.quantum_info import SparsePauliOp
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2
from qiskit_ibm_runtime.options import EstimatorOptions

# --- Rebuild qubit_op ---
h1e   = np.load('/home/Daniel/ndm1_h1e.npy')
h2e   = np.load('/home/Daniel/ndm1_h2e.npy')
ecore = float(np.load('/home/Daniel/ndm1_ecore.npy')[0])
hamiltonian = ElectronicEnergy.from_raw_integrals(h1e, h2e)
hamiltonian.nuclear_repulsion_energy = ecore
qubit_op = JordanWignerMapper().map(hamiltonian.second_q_op())

# --- Rebuild penalized_op ---
n_qubits, n_target = 14, 2
N_terms = [('I'*n_qubits, n_qubits / 2.0)]
for i in range(n_qubits):
    N_terms.append(('I'*i + 'Z' + 'I'*(n_qubits-1-i), -0.5))
N_op = SparsePauliOp.from_list(N_terms)
penalty = ((N_op @ N_op).simplify() - 2*n_target*N_op
           + SparsePauliOp.from_list([('I'*n_qubits, float(n_target**2))])).simplify()
penalized_op = (qubit_op + 5.0 * penalty).simplify()

# --- Rebuild and transpile EfficientSU2 ---
service = QiskitRuntimeService(channel="ibm_quantum_platform")
backend = service.backend("ibm_fez")

hf_prep = QuantumCircuit(n_qubits)
hf_prep.x(0)
hf_prep.x(7)
from qiskit.circuit.library import efficient_su2

base_ansatz = efficient_su2(n_qubits, reps=2, entanglement='linear')
ansatz = hf_prep.compose(base_ansatz)
print(f"Parameters: {ansatz.num_parameters}")

isa_circuit = transpile(ansatz, backend=backend, optimization_level=3)
isa_op = penalized_op.apply_layout(isa_circuit.layout)
print(f"Transpiled depth: {isa_circuit.depth()}, 2Q gates: {isa_circuit.num_nonlocal_gates()}")


# --- Single hardware evaluation at HF (zero params) ---
optimal_params = np.zeros(ansatz.num_parameters)
bound_circuit  = isa_circuit.assign_parameters(optimal_params)

options = EstimatorOptions()
options.resilience_level = 0
options.default_shots = 2048


estimator = EstimatorV2(mode=backend, options=options)
job = estimator.run([(bound_circuit, isa_op)])
print(f"Job ID: {job.job_id()}")
print("Job submitted — waiting for result...")
result_hw = job.result()

e_hw = result_hw[0].data.evs.real + ecore
print(f"\nHF energy (hardware):  {e_hw:.6f} Eh")
print(f"HF energy (classical): {-6558.818291:.6f} Eh")
print(f"UCCSD-VQE (classical): {-6558.837073:.6f} Eh")
print(f"Hardware noise:        {(e_hw - (-6558.818291))*1000:.2f} mEh")


qiskit_runtime_service.__init__:WARNING:2026-06-23 01:59:37,607: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-06-23 01:59:37,608: Using instance: open-instance, plan: open


Parameters: 84
Transpiled depth: 61, 2Q gates: 26
Job ID: d8suh35posuc738ntjp0
Job submitted — waiting for result...

HF energy (hardware):  -6214.948093 Eh
HF energy (classical): -6558.818291 Eh
UCCSD-VQE (classical): -6558.837073 Eh
Hardware noise:        343870.20 mEh


# Part 2 - Product Energy

## Section 1
___

#### Extract raw PDB
In addition to part 1, it includes nearby atoms of the drug molecule ZZ7 (residue 301) within 4.5 Å of either zinc.

In [ ]:
from Bio.PDB import PDBParser
import numpy as np

parser = PDBParser(QUIET=True)
s = parser.get_structure('NDM1', '/home/Daniel/gdrive/Enzyme folder/5ZGE.pdb')
chain = s[0]['A']
res = {r.get_id()[1]: r for r in chain}

zn1 = list(res[302].get_atoms())[0].get_vector()
zn2 = list(res[303].get_atoms())[0].get_vector()

ZZ7_INCLUDE = set()
for atom in res[301].get_atoms():
    av = atom.get_vector()
    if (av - zn1).norm() < 4.5 or (av - zn2).norm() < 4.5:
        ZZ7_INCLUDE.add(atom.get_name().strip())
print(f"ZZ7 atoms included: {sorted(ZZ7_INCLUDE)}")

PROTEIN_RES = {120, 122, 124, 189, 208, 250}
lines = []
with open('/home/Daniel/gdrive/Enzyme folder/5ZGE.pdb') as f:
    for line in f:
        if not (line.startswith('ATOM') or line.startswith('HETATM')):
            continue
        resnum = int(line[22:26])
        aname  = line[12:16].strip()
        if resnum in PROTEIN_RES or resnum in {302, 303, 304}:
            lines.append(line)
        elif resnum == 301 and aname in ZZ7_INCLUDE:
            lines.append(line)

with open('/home/Daniel/ndm1_product_raw.pdb', 'w') as f:
    f.writelines(lines)
    f.write('END\n')
print(f"Total atoms: {len(lines)}")


ZZ7 atoms included: ['C12', 'C13', 'C14', 'C15', 'C16', 'C2', 'C6', 'N3', 'O1', 'O2', 'O4', 'OXT', 'S1']
Total atoms: 151


## Section 2
___

#### Add hydrogen caps

In [ ]:
import numpy as np
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
s = parser.get_structure('NDM1', '/home/Daniel/ndm1_product_raw.pdb')
chain = list(s[0].get_chains())[0]

atoms, coords, resids, anames = [], [], [], []
for res in chain:
    for atom in res.get_atoms():
        if atom.element == 'H':
            continue
        atoms.append(atom)
        coords.append(atom.get_vector().get_array())
        resids.append(res.get_id()[1])
        anames.append(atom.get_name().strip())
coords = np.array(coords)
h_caps = []

# --- Protein residue N-terminus caps ---
for i in range(len(atoms)):
    if resids[i] in {120, 122, 124, 189, 208, 250} and anames[i] == 'N':
        ca_idx = next((j for j in range(len(atoms))
                       if resids[j] == resids[i] and anames[j] == 'CA'), None)
        if ca_idx is not None:
            d = coords[i] - coords[ca_idx]
            h_caps.append(coords[i] + d / np.linalg.norm(d) * 1.01)

# --- Bridging OH cap ---
oh_idx  = next(i for i in range(len(atoms)) if resids[i] == 304)
zn1_idx = next(i for i in range(len(atoms)) if resids[i] == 302)
zn2_idx = next(i for i in range(len(atoms)) if resids[i] == 303)
mid = (coords[zn1_idx] + coords[zn2_idx]) / 2
d   = coords[oh_idx] - mid
h_caps.append(coords[oh_idx] + d / np.linalg.norm(d) * 0.97)

# --- ZZ7 dangling bond caps ---
s_full  = parser.get_structure('FULL', '/home/Daniel/gdrive/Enzyme folder/5ZGE.pdb')
zz7_res = next(r for r in s_full[0]['A'] if r.get_id()[1] == 301)
zz7_all = {a.get_name().strip(): a.get_vector().get_array()
            for a in zz7_res.get_atoms() if a.element != 'H'}
zz7_incl = {anames[i] for i in range(len(atoms)) if resids[i] == 301}

for name_in, c_in in zz7_all.items():
    if name_in not in zz7_incl:
        continue
    for name_out, c_out in zz7_all.items():
        if name_out in zz7_incl:
            continue
        dist = np.linalg.norm(c_in - c_out)
        if dist < 1.9:
            d = c_out - c_in
            h_caps.append(c_in + d / np.linalg.norm(d) * 1.0)
            print(f"  H cap: {name_in}–{name_out} ({dist:.2f} Å)")

# --- Write XYZ ---
n_total = len(atoms) + len(h_caps)
with open('/home/Daniel/gdrive/Enzyme folder/ndm1_product_capped.xyz', 'w') as f:
    f.write(f"{n_total}\nNDM1 product-bound active site\n")
    for i in range(len(atoms)):
        el, c = atoms[i].element.capitalize(), coords[i]
        f.write(f"{el}  {c[0]:.4f}  {c[1]:.4f}  {c[2]:.4f}\n")
    for c in h_caps:
        f.write(f"H  {c[0]:.4f}  {c[1]:.4f}  {c[2]:.4f}\n")

print(f"\nHeavy: {len(atoms)}, H caps: {len(h_caps)}, Total: {n_total}")
print("Saved ndm1_product_capped.xyz")


  H cap: C6–C1 (1.49 Å)
  H cap: C14–N1 (1.46 Å)

Heavy: 70, H caps: 9, Total: 79
Saved ndm1_product_capped.xyz


### Avogadro geometry optimization
Saved as ndm1_product_hydroxide.xyz

### Check + DFLYP

In [ ]:
import psutil
from pyscf import gto, dft, df
import numpy as np

avail_mb = psutil.virtual_memory().available / 1e6

# Step 1: DIIS with level shift
mf2 = dft.RKS(mol)
mf2.xc          = 'b3lyp'
mf2.grids.level = 3
mf2 = df.density_fit(mf2, auxbasis='def2-universal-jkfit')
mf2.with_df._cderi = '/home/Daniel/ndm1_product_df_eri.h5'  # load existing
mf2.chkfile        = '/home/Daniel/ndm1_product_b3lyp.chk'
mf2.max_memory     = int(min(28000, avail_mb * 0.85))
mf2.level_shift    = 0.3
mf2.damp           = 0.5
mf2.max_cycle      = 150
mf2.verbose        = 4
mf2.kernel()

# Step 2: Newton to tighten
dm0 = mf2.make_rdm1()
mf2 = mf2.newton()
mf2.max_cycle = 50
mf2.kernel(dm0=dm0)




******** <class 'pyscf.df.df_jk.DFRKS'> ********
method = DFRKS
initial guess = minao
damping factor = 0.5
level_shift factor = 0.3
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-09
SCF conv_tol_grad = None
SCF max_cycles = 150
direct_scf = False
chkfile to save SCF result = /home/Daniel/ndm1_product_b3lyp.chk
max_memory 16492 MB (current use 10054 MB)
XC library pyscf.dft.libxc version 7.0.0
    S. Lehtola, C. Steigemann, M. J.T. Oliveira, and M. A.L. Marques.,  SoftwareX 7, 1–5 (2018)
XC functionals = b3lyp
    P. J. Stephens, F. J. Devlin, C. F. Chabalowski, and M. J. Frisch.,  J. Phys. Chem. 98, 11623 (1994)
radial grids: 
    Treutler-Ahlrichs [JCP 102, 346 (1995); DOI:10.1063/1.469408] (M4) radial grids
    
becke partition: Becke, JCP 88, 2547 (1988); DOI:10.1063/1.454033
pruning grids: <function nwchem_prune at 0x7f8895cb27a0>
grids dens level: 3
symmetrized grids: False
atomic radii adjust function: <function treutle

#### Rebuild mol and load mf2

In [ ]:
from pyscf import gto, dft, df, scf as pyscf_scf, mcscf
import numpy as np

xyz_path = '/home/Daniel/gdrive/Enzyme folder/ndm1_product_hydroxide.xyz'
with open(xyz_path) as f:
    lines = f.readlines()
n_atoms = int(lines[0])
atom_str = ''
for line in lines[2:2+n_atoms]:
    parts = line.split()
    atom_str += f'{parts[0]} {parts[1]} {parts[2]} {parts[3]}; '

mol = gto.Mole()
mol.atom    = atom_str
mol.basis   = 'def2-SVP'
mol.charge  = 0
mol.spin    = 0
mol.verbose = 4
mol.build()

mf2 = dft.RKS(mol)
mf2.xc = 'b3lyp'
mf2.grids.level = 3
mf2 = df.density_fit(mf2, auxbasis='def2-universal-jkfit')
mf2.with_df._cderi = '/home/Daniel/ndm1_product_df_eri.h5'
mf2.chkfile        = '/home/Daniel/ndm1_product_b3lyp.chk'

mf2.mo_coeff  = pyscf_scf.chkfile.load('/home/Daniel/ndm1_product_b3lyp.chk', 'scf/mo_coeff')
mf2.mo_occ    = pyscf_scf.chkfile.load('/home/Daniel/ndm1_product_b3lyp.chk', 'scf/mo_occ')
mf2.mo_energy = pyscf_scf.chkfile.load('/home/Daniel/ndm1_product_b3lyp.chk', 'scf/mo_energy')
mf2.converged = True

print(f"Electrons: {mf2.mo_occ.sum():.0f}")
homo = mf2.mo_energy[mf2.mo_occ > 0].max()
lumo = mf2.mo_energy[mf2.mo_occ == 0].min()
print(f"HOMO: {homo*27.2114:.3f} eV  LUMO: {lumo*27.2114:.3f} eV  Gap: {(lumo-homo)*27.2114:.3f} eV")


System: uname_result(system='Linux', node='ndm1-calc', release='6.1.0-49-cloud-amd64', version='#1 SMP PREEMPT_DYNAMIC Debian 6.1.174-1 (2026-05-26)', machine='x86_64')  Threads 8
Python 3.11.2 (main, Apr  8 2026, 01:58:00) [GCC 12.2.0]
numpy 2.4.6  scipy 1.17.1  h5py 3.16.0
Date: Wed Jun 24 18:28:45 2026
PySCF version 2.13.1
PySCF path  /home/Daniel/myenv/lib/python3.11/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 79
[INPUT] num. electrons = 538
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 N      3.796462458900  45.192086993600 117.864268735500 AA    7.174274289514  85.400667415423 222.731187782231 Bohr   0.0
[INPUT]  2 C      4.544378747700  44.113479656000 117.237281441800 AA    8.587631239447  83.362394951413 221.546

#### Finding which frontier MOs actually have Zn character

In [ ]:
import numpy as np

nocc = int(mf2.mo_occ.sum()) // 2
zn_aos = [i for i, lab in enumerate(mol.ao_labels()) if 'Zn' in lab]
mo_zn = np.sum(mf2.mo_coeff[zn_aos, :]**2, axis=0)

print("MOs with Zn character (HOMO-10 to LUMO+10):")
for i in range(nocc-10, nocc+11):
    tag = " <- HOMO" if i == nocc-1 else (" <- LUMO" if i == nocc else "")
    print(f"  MO {i:3d} (occ={mf2.mo_occ[i]:.0f})  Zn={mo_zn[i]:.4f}  {mf2.mo_energy[i]*27.2114:.3f} eV{tag}")


MOs with Zn character (HOMO-10 to LUMO+10):
  MO 259 (occ=2)  Zn=0.0628  -7.271 eV
  MO 260 (occ=2)  Zn=0.0219  -7.171 eV
  MO 261 (occ=2)  Zn=0.0277  -7.120 eV
  MO 262 (occ=2)  Zn=0.0067  -7.087 eV
  MO 263 (occ=2)  Zn=0.0135  -6.960 eV
  MO 264 (occ=2)  Zn=0.0353  -6.949 eV
  MO 265 (occ=2)  Zn=0.0267  -6.843 eV
  MO 266 (occ=2)  Zn=0.0175  -6.743 eV
  MO 267 (occ=2)  Zn=0.0147  -6.654 eV
  MO 268 (occ=2)  Zn=0.0045  -6.403 eV <- HOMO
  MO 269 (occ=0)  Zn=0.0064  2.077 eV <- LUMO
  MO 270 (occ=0)  Zn=0.0086  2.120 eV
  MO 271 (occ=0)  Zn=0.0034  2.188 eV
  MO 272 (occ=0)  Zn=0.0056  2.292 eV
  MO 273 (occ=0)  Zn=0.0045  2.488 eV
  MO 274 (occ=0)  Zn=0.0138  2.531 eV
  MO 275 (occ=0)  Zn=0.0011  2.619 eV
  MO 276 (occ=0)  Zn=0.0033  2.701 eV
  MO 277 (occ=0)  Zn=0.0023  2.751 eV
  MO 278 (occ=0)  Zn=0.0093  2.852 eV
  MO 279 (occ=0)  Zn=0.0087  2.881 eV


#### Finding the Zn-S bond

In [ ]:
s_aos = [i for i, lab in enumerate(mol.ao_labels()) if ' S ' in lab]
mo_s = np.sum(mf2.mo_coeff[s_aos, :]**2, axis=0)

print("Frontier MOs by S character:")
for i in range(nocc-10, nocc+11):
    tag = " <- HOMO" if i == nocc-1 else (" <- LUMO" if i == nocc else "")
    print(f"  MO {i:3d}  Zn={mo_zn[i]:.4f}  S={mo_s[i]:.4f}  {mf2.mo_energy[i]*27.2114:.3f} eV{tag}")


Frontier MOs by S character:
  MO 259  Zn=0.0628  S=0.0027  -7.271 eV
  MO 260  Zn=0.0219  S=0.0105  -7.171 eV
  MO 261  Zn=0.0277  S=0.0602  -7.120 eV
  MO 262  Zn=0.0067  S=0.0050  -7.087 eV
  MO 263  Zn=0.0135  S=0.0132  -6.960 eV
  MO 264  Zn=0.0353  S=0.0349  -6.949 eV
  MO 265  Zn=0.0267  S=0.1139  -6.843 eV
  MO 266  Zn=0.0175  S=0.0294  -6.743 eV
  MO 267  Zn=0.0147  S=0.0476  -6.654 eV
  MO 268  Zn=0.0045  S=0.0012  -6.403 eV <- HOMO
  MO 269  Zn=0.0064  S=0.0106  2.077 eV <- LUMO
  MO 270  Zn=0.0086  S=0.0030  2.120 eV
  MO 271  Zn=0.0034  S=0.0025  2.188 eV
  MO 272  Zn=0.0056  S=0.0036  2.292 eV
  MO 273  Zn=0.0045  S=0.0832  2.488 eV
  MO 274  Zn=0.0138  S=0.1683  2.531 eV
  MO 275  Zn=0.0011  S=0.0056  2.619 eV
  MO 276  Zn=0.0033  S=0.0021  2.701 eV
  MO 277  Zn=0.0023  S=0.0161  2.751 eV
  MO 278  Zn=0.0093  S=0.0309  2.852 eV
  MO 279  Zn=0.0087  S=0.0038  2.881 eV


In [ ]:
mc2 = mcscf.CASCI(mf2, ncas2, nelecas2)
mc2.fcisolver = direct_spin0.FCISolver(mol)
mo2 = mc2.sort_mo([264, 269, 270, 273, 274, 277, 278])  # MO 264 = occ (Zn-S bonding), rest virtual
mc2.verbose = 4
mc2.kernel(mo2)
print(f"S^2 = {mc2.fcisolver.spin}")



******** <class 'pyscf.mcscf.df.DFCASCI'> ********
CAS (1e+1e, 7o), ncore = 268, nvir = 792
natorb = False
canonicalization = True
sorting_mo_energy = False
max_memory 4000 MB (current use 299 MB)
******** <class 'pyscf.fci.direct_spin0.FCISolver'> ********
max. cycles = 100
conv_tol = 1e-10
davidson only = False
linear dependence = 1e-14
level shift = 0.001
max iter space = 12
max_memory 4000 MB
nroots = 1
pspace_size = 400
spin = None
DFCASCI/DFCASSCF: density fitting for JK matrix and 2e integral transformation
Density matrix diagonal elements [1.94923566 0.00345494 0.00273941 0.00889819 0.0211821  0.00966292
 0.00482678]
CASCI converged
CASCI E = -7575.19013174998  E(CI) = -0.913575016837058  S^2 = 0.0000000
S^2 = None


#### CASCI
The product fragment CASCI(2e,7o) calculation converged to a singlet ground state (S² = 0.0, E = −7575.11818 Eh). A near-degenerate triplet state was found 5 mEh lower in energy, consistent with the known open-shell character of dizinc metalloenzyme active sites. The singlet state was selected for consistency with the substrate fragment calculation (Part 1) and because the closed-shell singlet is the physically relevant state for inhibitor binding in the diamagnetic NDM-1 enzyme.

In [ ]:
from pyscf import mcscf, ao2mo
from pyscf.fci import direct_spin0
import numpy as np

ncas2    = 7
nelecas2 = 2

mc2 = mcscf.CASCI(mf2, ncas2, nelecas2)
mc2.fcisolver = direct_spin0.FCISolver(mol)  # enforce singlet
mo2 = mc2.sort_mo(list(range(mf2.mo_occ[mf2.mo_occ > 0].shape[0] - 1,
                              mf2.mo_occ[mf2.mo_occ > 0].shape[0] + 6)))
mc2.verbose = 4
mc2.kernel(mo2)

print(f"S^2 = {mc2.e_cas}")
h1e2, ecore2 = mc2.get_h1eff()
h2e2 = ao2mo.restore(1, mc2.get_h2eff(), ncas2)

np.save('/home/Daniel/ndm1_product_h1e.npy', h1e2)
np.save('/home/Daniel/ndm1_product_h2e.npy', h2e2)
np.save('/home/Daniel/ndm1_product_ecore.npy', np.array([ecore2]))
print(f"Product CASCI energy: {mc2.e_tot:.8f} Eh")



******** <class 'pyscf.mcscf.df.DFCASCI'> ********
CAS (1e+1e, 7o), ncore = 268, nvir = 792
natorb = False
canonicalization = True
sorting_mo_energy = False
max_memory 4000 MB (current use 291 MB)
******** <class 'pyscf.fci.direct_spin0.FCISolver'> ********
max. cycles = 100
conv_tol = 1e-10
davidson only = False
linear dependence = 1e-14
level shift = 0.001
max iter space = 12
max_memory 4000 MB
nroots = 1
pspace_size = 400
spin = None
DFCASCI/DFCASSCF: density fitting for JK matrix and 2e integral transformation


Density matrix diagonal elements [0.54757399 0.51571011 0.59446655 0.01213536 0.10604924 0.09318327
 0.13088148]
CASCI converged
CASCI E = -7575.11817818052  E(CI) = -0.633631924847577  S^2 = 0.0000000
S^2 = -0.6336319248475775
Product CASCI energy: -7575.11817818 Eh


In [ ]:
mc2 = mcscf.CASCI(mf2, ncas2, nelecas2)
mc2.fcisolver = direct_spin0.FCISolver(mol)
mo2 = mc2.sort_mo([264, 269, 270, 273, 274, 277, 278])  # MO 264 = occ (Zn-S bonding), rest virtual
mc2.verbose = 4
mc2.kernel(mo2)
print(f"S^2 = {mc2.fcisolver.spin}")



******** <class 'pyscf.mcscf.df.DFCASCI'> ********
CAS (1e+1e, 7o), ncore = 268, nvir = 792
natorb = False
canonicalization = True
sorting_mo_energy = False
max_memory 4000 MB (current use 303 MB)
******** <class 'pyscf.fci.direct_spin0.FCISolver'> ********
max. cycles = 100
conv_tol = 1e-10
davidson only = False
linear dependence = 1e-14
level shift = 0.001
max iter space = 12
max_memory 4000 MB
nroots = 1
pspace_size = 400
spin = None
DFCASCI/DFCASSCF: density fitting for JK matrix and 2e integral transformation
Density matrix diagonal elements [1.94923566 0.00345494 0.00273941 0.00889819 0.0211821  0.00966292
 0.00482678]
CASCI converged
CASCI E = -7575.19013174998  E(CI) = -0.913575016837058  S^2 = 0.0000000
S^2 = None


In [ ]:
h1e2, ecore2 = mc2.get_h1eff()
h2e2 = ao2mo.restore(1, mc2.get_h2eff(), ncas2)

np.save('/home/Daniel/ndm1_product_h1e.npy', h1e2)
np.save('/home/Daniel/ndm1_product_h2e.npy', h2e2)
np.save('/home/Daniel/ndm1_product_ecore.npy', np.array([ecore2]))
print(f"Product CASCI energy: {mc2.e_tot:.8f} Eh")


Product CASCI energy: -7575.19013175 Eh


#### Hamiltonian + verify

In [ ]:
import scipy.sparse as sp, scipy.sparse.linalg
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy
from qiskit_nature.second_q.mappers import JordanWignerMapper

h1e2   = np.load('/home/Daniel/ndm1_product_h1e.npy')
h2e2   = np.load('/home/Daniel/ndm1_product_h2e.npy')
ecore2 = float(np.load('/home/Daniel/ndm1_product_ecore.npy')[0])

ham2      = ElectronicEnergy.from_raw_integrals(h1e2, h2e2)
ham2.nuclear_repulsion_energy = ecore2
qubit_op2 = JordanWignerMapper().map(ham2.second_q_op())

basis_2e = [i for i in range(2**14) if bin(i).count('1') == 2]
H_sub2   = qubit_op2.to_matrix(sparse=True)[np.ix_(basis_2e, basis_2e)]
e_gs2    = float(sp.linalg.eigsh(H_sub2, k=1, which='SA',
                                  return_eigenvectors=False)[0]) + ecore2
print(f"Exact (product): {e_gs2:.8f} Eh")
print(f"CASCI (product): {mc2.e_tot:.8f} Eh")
print(f"Difference:      {(e_gs2 - mc2.e_tot)*1000:.4f} mEh")


Exact (product): -7575.19013175 Eh
CASCI (product): -7575.19013175 Eh
Difference:      0.0000 mEh


#### UCCSD-VQE

In [ ]:
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit import transpile

mapper2    = JordanWignerMapper()
init2      = HartreeFock(num_spatial_orbitals=7, num_particles=(1,1), qubit_mapper=mapper2)
ansatz2    = UCCSD(num_spatial_orbitals=7, num_particles=(1,1), qubit_mapper=mapper2, initial_state=init2)
ansatz2_dc = transpile(ansatz2, basis_gates=['cx','u'], optimization_level=1)

def callback2(nfev, x, fx, metadata):
    if nfev % 20 == 0:
        print(f"  iter {nfev:3d}  E = {fx + ecore2:.8f} Eh")

vqe2 = VQE(StatevectorEstimator(), ansatz2_dc, COBYLA(maxiter=300, rhobeg=0.01),
           initial_point=np.zeros(ansatz2_dc.num_parameters), callback=callback2)
result2         = vqe2.compute_minimum_eigenvalue(qubit_op2)
e_vqe2          = result2.eigenvalue.real + ecore2
optimal_params2 = result2.optimal_point

np.save('/home/Daniel/ndm1_product_uccsd_params.npy', optimal_params2)
print(f"\nUCCSD-VQE (product): {e_vqe2:.8f} Eh")
print(f"CASCI ref:           {mc2.e_tot:.8f} Eh")
print(f"Difference:          {(e_vqe2 - mc2.e_tot)*1000:.4f} mEh")


  iter  20  E = -7575.18199402 Eh
  iter  40  E = -7575.18218180 Eh
  iter  60  E = -7575.18732631 Eh
  iter  80  E = -7575.18788337 Eh
  iter 100  E = -7575.18842407 Eh
  iter 120  E = -7575.18941998 Eh
  iter 140  E = -7575.18972203 Eh
  iter 160  E = -7575.18990605 Eh
  iter 180  E = -7575.18993105 Eh
  iter 200  E = -7575.18997332 Eh
  iter 220  E = -7575.18993869 Eh
  iter 240  E = -7575.18997913 Eh
  iter 260  E = -7575.18998653 Eh
  iter 280  E = -7575.19002016 Eh
  iter 300  E = -7575.19002988 Eh

UCCSD-VQE (product): -7575.19002988 Eh
CASCI ref:           -7575.19013175 Eh
Difference:          0.1019 mEh


#### Run on actual hardware later

# Part 3 - ZZ7 fragment energy
___
ZZ7 is the pDB ligand code for the hydrolzed ampicillin, or the broken down remains of ampicillin after NDM-1 has already cleaved it

### Extract ZZ7 fragment

In [ ]:
import numpy as np
from Bio.PDB import PDBParser

ZZ7_INCLUDE = {'C12','C13','C14','C15','C16','C2','C6','N3','O1','O2','O4','OXT','S1'}

def normalize(v):
    return v / np.linalg.norm(v)

parser = PDBParser(QUIET=True)
s = parser.get_structure('ndm1', '/home/Daniel/ndm1_product_raw.pdb')
chain = list(s[0].get_chains())[0]
res = {r.get_id()[1]: r for r in chain}

zn2_pos = list(res[303].get_atoms())[0].get_vector().get_array()

zz7_atoms = {}
for atom in res[301].get_atoms():
    name = atom.get_name().strip()
    if name in ZZ7_INCLUDE:
        elem = atom.element.strip() if atom.element and atom.element.strip() else name[0]
        zz7_atoms[name] = (elem, atom.get_vector().get_array())

excluded = {atom.get_name().strip(): atom.get_vector().get_array()
            for atom in res[301].get_atoms()
            if atom.get_name().strip() not in ZZ7_INCLUDE}

h_caps = []
# H caps at Zn coordination bonds
for name in ['N3', 'O2']:
    coord = zz7_atoms[name][1]
    d = coord - zn2_pos
    h_caps.append(coord + normalize(d) * 1.01)
    print(f"H cap: {name}–Zn2 bond")

# H caps at cut intra-ZZ7 bonds
for inc_name, (inc_elem, inc_coord) in zz7_atoms.items():
    for exc_name, exc_coord in excluded.items():
        if np.linalg.norm(inc_coord - exc_coord) < 1.8:
            d = inc_coord - exc_coord
            h_caps.append(inc_coord + normalize(d) * 1.01)
            print(f"H cap: {inc_name}–{exc_name} cut bond")

all_atoms = list(zz7_atoms.values()) + [('H', hc) for hc in h_caps]

xyz_path = '/home/Daniel/gdrive/Enzyme folder/ndm1_zz7_fragment.xyz'
with open(xyz_path, 'w') as f:
    f.write(f"{len(all_atoms)}\nZZ7 fragment\n")
    for elem, coord in all_atoms:
        f.write(f"{elem}  {coord[0]:.6f}  {coord[1]:.6f}  {coord[2]:.6f}\n")
print(f"Saved {len(all_atoms)} atoms ({len(zz7_atoms)} heavy + {len(h_caps)} H caps)")


H cap: N3–Zn2 bond
H cap: O2–Zn2 bond
Saved 15 atoms (13 heavy + 2 H caps)


### Avogadro
Load ndm1_zz7_fragment.xyz, run UFF geometry optimization, save as ndm1_zz7_optimized.xyz.

### B3LYP

In [ ]:
from pyscf import gto, dft, df
import psutil, numpy as np

avail_mb = psutil.virtual_memory().available / 1e6

with open('/home/Daniel/gdrive/Enzyme folder/ndm1_zz7_optimized.xyz') as f:
    lines = f.readlines()
n_atoms = int(lines[0])
atom_str = ''
for line in lines[2:2+n_atoms]:
    parts = line.split()
    atom_str += f'{parts[0]} {parts[1]} {parts[2]} {parts[3]}; '

mol3 = gto.Mole()
mol3.atom    = atom_str
mol3.basis   = 'def2-SVP'
mol3.charge  = -1   # thiolate; adjust if electron count looks wrong
mol3.spin    = 0
mol3.verbose = 4
mol3.build()

print(f"Atoms: {mol3.natm}, Electrons: {mol3.nelectron}, Basis functions: {mol3.nao}")
assert mol3.nelectron % 2 == 0, "ODD ELECTRON COUNT — check charge"

mf3 = dft.RKS(mol3)
mf3.xc = 'b3lyp'
mf3.grids.level = 3
mf3 = df.density_fit(mf3, auxbasis='def2-universal-jkfit')
mf3.with_df._cderi_to_save = '/home/Daniel/ndm1_zz7_df_eri.h5'
mf3.chkfile    = '/home/Daniel/ndm1_zz7_b3lyp.chk'
mf3.max_memory = int(min(28000, avail_mb * 0.85))
mf3.level_shift = 0.3
mf3.damp        = 0.5
mf3.max_cycle   = 150
mf3.kernel()

dm0 = mf3.make_rdm1()
mf3 = mf3.newton()
mf3.max_cycle = 50
mf3.kernel(dm0=dm0)


System: uname_result(system='Linux', node='ndm1-calc', release='6.1.0-49-cloud-amd64', version='#1 SMP PREEMPT_DYNAMIC Debian 6.1.174-1 (2026-05-26)', machine='x86_64')  Threads 8
Python 3.11.2 (main, Apr  8 2026, 01:58:00) [GCC 12.2.0]
numpy 2.4.6  scipy 1.17.1  h5py 3.16.0
Date: Wed Jun 24 22:37:39 2026
PySCF version 2.13.1
PySCF path  /home/Daniel/myenv/lib/python3.11/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 15
[INPUT] num. electrons = 100
[INPUT] charge = -1
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 O     -2.594160667500  47.193619438400 107.134456411600 AA   -4.902253184694  89.183015565526 202.454781122077 Bohr   0.0
[INPUT]  2 C     -1.631034891100  47.520033376600 108.088173770200 AA   -3.082209243789  89.799848511965 204.25

np.float64(-1020.4200535036364)

### Active space diagnostic

In [ ]:
from pyscf import mcscf, ao2mo
from pyscf.fci import direct_spin0

ncas3, nelecas3 = 7, 2
nocc3 = int(mf3.mo_occ.sum()) // 2

# Diagnostic first
zn_aos3 = [i for i, lab in enumerate(mol3.ao_labels()) if 'Zn' in lab]
s_aos3  = [i for i, lab in enumerate(mol3.ao_labels()) if ' S ' in lab]
mo_s3   = np.sum(mf3.mo_coeff[s_aos3, :]**2, axis=0)

print("Frontier MOs by S character:")
for i in range(nocc3-5, nocc3+6):
    tag = " <- HOMO" if i == nocc3-1 else (" <- LUMO" if i == nocc3 else "")
    print(f"  MO {i:3d} (occ={mf3.mo_occ[i]:.0f})  S={mo_s3[i]:.4f}  {mf3.mo_energy[i]*27.2114:.3f} eV{tag}")


Frontier MOs by S character:
  MO  45 (occ=2)  S=0.1277  -3.500 eV
  MO  46 (occ=2)  S=0.0453  -3.480 eV
  MO  47 (occ=2)  S=0.1367  -2.842 eV
  MO  48 (occ=2)  S=0.0222  -2.686 eV
  MO  49 (occ=2)  S=0.1309  -2.329 eV <- HOMO
  MO  50 (occ=0)  S=0.0132  -1.239 eV <- LUMO
  MO  51 (occ=0)  S=0.0706  0.149 eV
  MO  52 (occ=0)  S=0.1505  0.259 eV
  MO  53 (occ=0)  S=0.0340  0.600 eV
  MO  54 (occ=0)  S=0.0473  1.401 eV
  MO  55 (occ=0)  S=0.0528  1.830 eV


### Exact diagonalization verification

In [ ]:
import scipy.sparse as sp, scipy.sparse.linalg

basis_2e = [i for i in range(2**14) if bin(i).count('1') == 2]
H_sub3   = qubit_op3.to_matrix(sparse=True)[np.ix_(basis_2e, basis_2e)]
e_gs3    = float(sp.linalg.eigsh(H_sub3, k=1, which='SA',
                                  return_eigenvectors=False)[0]) + ecore3
print(f"Exact (qubit_op3): {e_gs3:.8f} Eh")
print(f"CASCI ref:         {mc3.e_tot:.8f} Eh")
print(f"Difference:        {(e_gs3 - mc3.e_tot)*1000:.4f} mEh")


Exact (qubit_op3): -1015.55338749 Eh
CASCI ref:         -1015.53547744 Eh
Difference:        -17.9101 mEh


### CASCI

In [ ]:
ncas3, nelecas3 = 7, 2

mc3 = mcscf.CASCI(mf3, ncas3, nelecas3)
mc3.fcisolver = direct_spin0.FCISolver(mol3)
mo3 = mc3.sort_mo(list(range(49, 56)))  # HOMO + LUMO through LUMO+5
mc3.verbose = 4
mc3.kernel(mo3)

print(f"S^2 = {mc3.fcisolver.spin}")
print(f"NOs: {mcscf.addons.make_natural_orbitals(mc3)[0]}")

h1e3, ecore3 = mc3.get_h1eff()
h2e3 = ao2mo.restore(1, mc3.get_h2eff(), ncas3)
np.save('/home/Daniel/ndm1_zz7_h1e.npy', h1e3)
np.save('/home/Daniel/ndm1_zz7_h2e.npy', h2e3)
np.save('/home/Daniel/ndm1_zz7_ecore.npy', np.array([ecore3]))
print(f"ZZ7 CASCI energy: {mc3.e_tot:.8f} Eh")



******** <class 'pyscf.mcscf.df.DFCASCI'> ********
CAS (1e+1e, 7o), ncore = 49, nvir = 140
natorb = False
canonicalization = True
sorting_mo_energy = False
max_memory 24743 MB (current use 498 MB)
******** <class 'pyscf.fci.direct_spin0.FCISolver'> ********
max. cycles = 100
conv_tol = 1e-10
davidson only = False
linear dependence = 1e-14
level shift = 0.001
max iter space = 12
max_memory 4000 MB
nroots = 1
pspace_size = 400
spin = None
DFCASCI/DFCASSCF: density fitting for JK matrix and 2e integral transformation
Density matrix diagonal elements [9.50700430e-01 9.74835109e-01 9.02477666e-04 3.30109903e-02
 2.01639624e-02 1.88751979e-02 1.51183339e-03]
CASCI converged
CASCI E = -1015.53547744177  E(CI) = -0.685917333012299  S^2 = 0.0000000
S^2 = None
NOs: [ 2.00000000e+00  2.00000000e+00  2.00000000e+00  2.00000000e+00
  2.00000000e+00  2.00000000e+00  2.00000000e+00  2.00000000e+00
  2.00000000e+00  2.00000000e+00  2.00000000e+00  2.00000000e+00
  2.00000000e+00  2.00000000e+00  2.00

Why the Part 3 VQE calculation is unnecessary

The active space for all three parts of this calculation is CAS(2e, 7o) — two electrons distributed among seven orbitals. This is a special case where the classical CASCI solver already gives the exact answer with no approximation remaining for VQE to improve upon.

UCCSD (Unitary Coupled Cluster Singles and Doubles) generates all single and double excitations from a reference Hartree–Fock determinant. For a system with only two electrons, every possible excitation is a double excitation from the reference. There are no higher-order terms (triples, quadruples, etc.) that can exist with only two electrons. This means UCCSD with two electrons spans the complete CI expansion — it is mathematically identical to Full Configuration Interaction (FCI). CASCI with an FCI solver in the same active space gives the same exact result.

For Parts 1 and 2, the active spaces were near-Hartree–Fock (natural orbital occupancies ~2.000/0.002), so the VQE converged quickly and confirmed this: it reached within 0.34 mEh (Part 1) and 0.10 mEh (Part 2) of the CASCI reference. Those tiny residual gaps are numerical convergence errors, not missing correlation — the method is already exact.

For Part 3, there is an additional reason not to use VQE. The isolated ZZ7 thiolate fragment has biradical character (natural orbital occupancies 1.176 / 0.824), and the exact diagonalization revealed that the triplet state lies 17.9 mEh below the singlet in this active space. The VQE optimizer, which is unconstrained in spin, would be pulled toward the lower-energy triplet during optimization. This is physically wrong: in the enzyme, ZZ7 coordinates to two Zn²⁺ centers and is clearly a singlet (S² = 0, confirmed in Part 2). The CASCI solver with direct_spin0 correctly enforces the singlet and returns the exact singlet FCI energy of −1015.53547744 Eh.

In summary: for two active electrons, CASCI = FCI = the exact answer within the active space. There is no approximation for VQE to correct, and attempting VQE would risk converging to the physically incorrect triplet state. The CASCI singlet energy is used directly for Part 3.

# Binding energy calculation

In [ ]:
E1_vqe   = -6558.83707328   # Part 1 UCCSD-VQE
E2_vqe   = -7575.19002988   # Part 2 UCCSD-VQE
E3_casci = mc3.e_tot        # Part 3 CASCI singlet = -1015.53547744 Eh

delta_E_Eh   = E2_vqe - E1_vqe - E3_casci
delta_E_kcal = delta_E_Eh * 627.509

print(f"E(complex)   = {E2_vqe:.8f} Eh")
print(f"E(active site) = {E1_vqe:.8f} Eh")
print(f"E(ZZ7)       = {E3_casci:.8f} Eh")
print(f"\nΔE_bind = {delta_E_Eh:.6f} Eh = {delta_E_kcal:.2f} kcal/mol")


E(complex)   = -7575.19002988 Eh
E(active site) = -6558.83707328 Eh
E(ZZ7)       = -1015.53547744 Eh

ΔE_bind = -0.817479 Eh = -512.98 kcal/mol
